# Phase 1: Exploring the World Bank API

**Goal:** understand the source data *before* writing the pipeline. By the end you should be able to explain what one record means, how paging works, and which indicators have gaps.

**How to use this notebook:** run cells top to bottom with **Shift + Enter**. Read the markdown under each output: it tells you what to look for. Answer the questions in the *Your notes* cell at the end.

Kernel check: the top-right of Jupyter should say **Python (econdev)**. If not, use *Kernel → Change kernel*.

## 1. Setup

We read countries, years and indicator codes from `config/indicators.yml` instead of typing them here. The ingestion script will read the same file, so the notebook and the pipeline can never disagree about scope.

In [2]:
import json
from pathlib import Path

import pandas as pd
import requests
import yaml

config = yaml.safe_load(Path("../config/indicators.yml").read_text())
COUNTRIES = config["countries"]
START, END = config["start_year"], config["end_year"]
INDICATORS = config["indicators"]          # {code: friendly_name}

BASE_URL = "https://api.worldbank.org/v2"
print("Countries:", COUNTRIES)
print("Years:", START, "to", END)
print("Indicators:", INDICATORS)

Countries: ['NGA', 'KEN', 'GHA', 'RWA', 'ZAF']
Years: 2000 to 2025
Indicators: {'NY.GDP.MKTP.KD.ZG': 'gdp_growth_pct', 'SI.POV.DDAY': 'poverty_pct', 'IT.CEL.SETS.P2': 'mobile_per_100', 'SL.UEM.TOTL.ZS': 'unemployment_pct'}


## 2. One request, raw

An API call is just an HTTP GET to a URL with some parameters. Let's make one and look at what comes back **before** turning it into a table.

In [3]:
url = f"{BASE_URL}/country/{';'.join(COUNTRIES)}/indicator/NY.GDP.MKTP.KD.ZG"
params = {"format": "json", "per_page": 1000, "date": f"{START}:{END}"}

resp = requests.get(url, params=params, timeout=30)
print("Status code:", resp.status_code)     # 200 = OK
print("Full URL:", resp.url)

body = resp.json()
print("Top-level type:", type(body).__name__, "with", len(body), "items")

Status code: 200
Full URL: https://api.worldbank.org/v2/country/NGA;KEN;GHA;RWA;ZAF/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=1000&date=2000%3A2025
Top-level type: list with 2 items


**What to notice:** the response is a list of **2 items**, not a single object. That's a World Bank quirk. Item 0 is page metadata, item 1 is the actual records.

In [4]:
meta, records = body
meta

{'page': 1,
 'pages': 1,
 'per_page': 1000,
 'total': 130,
 'sourceid': '2',
 'lastupdated': '2026-07-13'}

**What to notice:** `total` is the number of records across all pages (5 countries × 26 years = 130). `pages` tells you whether you need to ask for more. `lastupdated` is when the World Bank last revised this indicator: worth keeping, because past values can change.

## 3. One record up close

Before writing any SQL you need to know the **grain**: what does one record represent?

In [5]:
print(json.dumps(records[0], indent=2))

{
  "indicator": {
    "id": "NY.GDP.MKTP.KD.ZG",
    "value": "GDP growth (annual %)"
  },
  "country": {
    "id": "GH",
    "value": "Ghana"
  },
  "countryiso3code": "GHA",
  "date": "2025",
  "value": 5.95441714756899,
  "unit": "",
  "obs_status": "",
  "decimal": 1
}


**What to notice:**
- One record = **one country, one indicator, one year**. That's the grain of your staging model.
- `indicator` and `country` are *nested* objects (each has an `id` and a `value`).
- `date` is a **string** (`"2025"`), not a number. Staging will need to cast it.
- `country.id` is a 2-letter code but `countryiso3code` is 3 letters. Pick one and use it everywhere (the config uses 3-letter codes).
- `value` can be `null`. Run the next cell to see how often.

In [6]:
df = pd.json_normalize(records)    # flattens nested JSON: country.value -> 'country.value' column
print(df.dtypes, "\n")
df.head()

countryiso3code        str
date                   str
value              float64
unit                   str
obs_status             str
decimal              int64
indicator.id           str
indicator.value        str
country.id             str
country.value          str
dtype: object 



,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,GHA,2025,5.954417,,,1,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),GH,Ghana
1,GHA,2024,5.847469,,,1,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),GH,Ghana
2,GHA,2023,3.140091,,,1,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),GH,Ghana
3,GHA,2022,3.804850,,,1,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),GH,Ghana
4,GHA,2021,5.076466,,,1,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),GH,Ghana


**What to notice:** `pd.json_normalize` turned the nested objects into dotted column names like `country.value`. In BigQuery you'll do the same thing with `JSON_VALUE(payload, '$.country.value')`. Also check the dtypes: `date` came through as `str` (text), not a number.

## 4. Pagination

APIs rarely return everything in one go. Here we deliberately ask for small pages to see how it works.

In [7]:
small = requests.get(url, params={**params, "per_page": 50}, timeout=30).json()
small[0]

{'page': 1,
 'pages': 3,
 'per_page': 50,
 'total': 130,
 'sourceid': '2',
 'lastupdated': '2026-07-13'}

Now `pages` is 3. To get everything you must request `page=1`, `page=2`, `page=3` and combine them. The loop below is exactly what `worldbank_client.py` will do in Phase 2.

In [8]:
def fetch_all_pages(url, params):
    records, page = [], 1
    while True:
        meta, rows = requests.get(url, params={**params, "page": page}, timeout=30).json()
        records.extend(rows or [])
        print(f"page {page} of {meta['pages']}: {len(rows or [])} records")
        if page >= meta["pages"]:
            return records
        page += 1


paged = fetch_all_pages(url, {**params, "per_page": 50})
print("Total collected:", len(paged))

page 1 of 3: 50 records
page 2 of 3: 50 records
page 3 of 3: 30 records
Total collected: 130


**Check:** the total should match `meta["total"]` from section 2. If a pipeline forgets to page, it silently loads only the first page, and nothing errors. That's one of the most common ingestion bugs.

## 5. All four indicators: how complete is the data?

Now pull every indicator and count how many years actually have a value. This **coverage table** is one of the most useful things you can make when you meet a new dataset.

In [9]:
frames = []
for code, name in INDICATORS.items():
    recs = fetch_all_pages(f"{BASE_URL}/country/{';'.join(COUNTRIES)}/indicator/{code}",
                           {**params, "per_page": 1000})
    frame = pd.json_normalize(recs)
    frame["indicator_name"] = name
    frames.append(frame)

all_df = pd.concat(frames, ignore_index=True)
print("Rows:", len(all_df), "(expect 5 countries x 4 indicators x 26 years = 520)")

coverage = (all_df.assign(has_value=all_df["value"].notna())
                  .pivot_table(index="countryiso3code", columns="indicator_name",
                               values="has_value", aggfunc="sum"))
coverage

page 1 of 1: 130 records
page 1 of 1: 130 records
page 1 of 1: 130 records
page 1 of 1: 130 records
Rows: 520 (expect 5 countries x 4 indicators x 26 years = 520)


indicator_name,gdp_growth_pct,mobile_per_100,poverty_pct,unemployment_pct
countryiso3code,,,,
GHA,26,25,3,26
KEN,26,25,5,26
NGA,26,25,6,26
RWA,26,25,6,26
ZAF,26,25,6,26


**What to notice:** three indicators have 25–26 years per country, but `poverty_pct` has only a handful. That gap shapes the whole project: you can't compare poverty year by year, only between survey years.

## 6. Poverty deep dive

This is the cell you wrote: only the years where a poverty survey actually exists.

In [10]:
url = "https://api.worldbank.org/v2/country/NGA;KEN;GHA;RWA;ZAF/indicator/SI.POV.DDAY"
meta, records = requests.get(url, params={"format": "json", "per_page": 1000, "date": "2000:2025"}).json()

df = pd.json_normalize(records)                      # flattens nested JSON into columns
df[df["value"].notna()][["countryiso3code", "date", "value"]]   # only the years with real data

,countryiso3code,date,value
9,GHA,2016,39.0
13,GHA,2012,41.8
20,GHA,2005,60.9
29,KEN,2022,45.5
30,KEN,2021,46.4
31,KEN,2020,44.4
36,KEN,2015,37.7
46,KEN,2005,44.6
55,NGA,2022,41.8
59,NGA,2018,34.2


**What to notice:** the survey years differ by country (Ghana's latest is 2016, Rwanda's is 2023). To compare countries fairly, the analysis will look at the *change* between two survey years against average GDP growth over the same span.

## 7. What does each indicator actually measure?

The API also describes each indicator. Read these: they go straight into your dbt column descriptions and README.

In [11]:
for code in INDICATORS:
    info = requests.get(f"{BASE_URL}/indicator/{code}", params={"format": "json"}, timeout=30).json()[1][0]
    print(f"{code}: {info['name']}")
    print(f"  Source: {info['sourceOrganization'][:150]}")
    print(f"  Note: {info['sourceNote'][:300]}...\n")

NY.GDP.MKTP.KD.ZG: GDP growth (annual %)
  Source: Country official statistics, National Statistical Organizations and/or Central Banks;
National Accounts data files, Organisation for Economic Co-opera
  Note: Gross domestic product is the total income earned through the production of goods and services in an economic territory during an accounting period. It can be measured in three different ways: using either the expenditure approach, the income approach, or the production approach. This indicator deno...

SI.POV.DDAY: Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)
  Source: World Bank, Poverty and Inequality Platform. Data are based on primary household survey data obtained from government statistical agencies and World B
  Note: Poverty headcount ratio at $3.00 a day is the percentage of the population living on less than $3.00 a day at 2021 purchasing power adjusted prices. As a result of revisions in PPP exchange rates, poverty rates for individual countries

## 8. Your notes

Double-click this cell and answer in your own words. These become the *Data* section of your README.

1. What does one record represent (the grain)?
2. Which fields need cleaning or type casting in staging?
3. Which indicator has gaps, and how will you handle them?
4. Why does pagination matter, even though our data fits on one page?
5. Anything that surprised you?

My answers:
1. The grain is one indicator, one country, one year
2. The year need to be cast as interger
3. Poverty has gap, i think we will assume the same poverty rate may need to be repeated across years up to the time it changes, same for mobile per 100
4. pagination is important in order to load all the data
5. The incomplete data perhaps of some indicators like poverty rate and mobile per 100